# Modelado: preprocesamiento + selección de variables × 4 modelos

Un estudio de Optuna **por modelo** (ridge, lasso, mlp, xgboost). Cada trial elige a la vez los
hiperparámetros del modelo y la configuración del pipeline:

`transformador` → `scaler` → `selección de variables` → `modelo`

- Se evalúa con `TimeSeriesSplit` **sobre train**; después de cada fold se reporta el MAE medio acumulado y el
  `MedianPruner` corta los trials que van peor que la mediana en ese fold (no se desperdician folds en configs malas).
- La mejor configuración se reentrena en todo train y se mide en **validation**.
- Al final se compara contra los baselines (persistencia y media móvil 7 días), medidos en los mismos folds y en val.
- Test **no se carga** en este notebook.

`dataset_final.csv` ya viene con inválidos, outliers (IQR clip) e imputación KNN aplicados (`src/main.py`);
la transformación, el escalado y la selección se ajustan dentro de cada fold, acá.

## Configuración

In [ ]:
from __future__ import annotations

import json
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Callable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from optuna.visualization import matplotlib as optuna_plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import RFE, SelectFromModel, mutual_info_regression
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler, PowerTransformer, QuantileTransformer,
                                   RobustScaler, StandardScaler)
from xgboost import XGBRegressor

# raíz del proyecto = la primera carpeta (desde acá hacia arriba) que tiene src/preprocess.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "preprocess.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from preprocess import COLS_CICLICAS, RUTA_FINAL, TARGET, huella

# ------------------------------------------------------------------ estudio
MODELOS = ["ridge", "lasso", "mlp", "xgboost"]
N_TRIALS = 60        # trials por modelo
N_SPLITS = 5         # folds de TimeSeriesSplit sobre train
TIMEOUT = 20 * 60    # segundos máximos por modelo (None = sin límite)
SEMILLA = 42

# ------------------------------------------------------------------ rutas
RUTA_LOGS = ROOT / "results" / "experiment_logs.csv"
RUTA_OPTUNA_DB = ROOT / "results" / "optuna.db"   # estudios persistidos: optuna-dashboard la puede abrir
RUTA_CACHE = ROOT / "results" / ".cache"          # transformaciones y scores ya calculados (por fold)
RUTA_OPTUNA_DB.parent.mkdir(parents=True, exist_ok=True)
memoria = joblib.Memory(RUTA_CACHE, verbose=0)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

## Datos

In [ ]:
def cargar_datos(ruta: str | Path = RUTA_FINAL, incluir_test: bool = False) -> pd.DataFrame:
    """Lee dataset_final.csv indexado por fecha. Por defecto descarta las filas de test:
    ni siquiera quedan en memoria. Pasar incluir_test=True solo para la evaluación final."""
    df = pd.read_csv(ruta, parse_dates=["fecha"]).sort_values("fecha").set_index("fecha")
    if not incluir_test:
        df = df[df["split"] != "test"]
    return df


def particion(df: pd.DataFrame, split: str) -> tuple[pd.DataFrame, pd.Series]:
    """(X, y) de un split. Features = todo menos el target y la columna `split`."""
    d = df[df["split"] == split]
    return d.drop(columns=[TARGET, "split"]), d[TARGET]


def _metricas(y, pred) -> dict:
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    ok = ~np.isnan(pred)  # los primeros días de un rolling quedan en NaN
    y, pred = y[ok], pred[ok]
    return {
        "mae": float(mean_absolute_error(y, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y, pred))),
        "r2": float(r2_score(y, pred)),
    }

## Baselines

In [ ]:
def predicciones_baselines(df: pd.DataFrame) -> pd.DataFrame:
    """Predicción de cada baseline para la fila D (que apunta al consumo de D+1).

    El target de la fila D es frio(D+1), así que frio(D) = target de la fila D-1:
      persistencia   : frio(D)                      = y.shift(1)
      media móvil 7d : promedio de frio(D-6 .. D)   = y.shift(1).rolling(7).mean()
    Solo usa información disponible en D: no hay fuga hacia el futuro."""
    if not (df.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all():
        raise ValueError("Hay días faltantes en dataset_final.csv: shift(1) ya no equivale a 'ayer'.")
    y = df[TARGET]
    return pd.DataFrame({
        "baseline_persistencia": y.shift(1),
        "baseline_media_movil_7d": y.shift(1).rolling(7).mean(),
    })

## Pipeline

| Paso | Opciones que prueba Optuna |
|---|---|
| Transformador | `yeo-johnson` / `quantile` (a normal) / `ninguno` |
| Scaler | `standard` / `robust` / `minmax` / `ninguno` |
| Selección | `filtro` (top-k por \|corr\| o información mutua, sin redundantes) / `filtro+rfe` (wrapper sobre las k del filtro) / `lasso` (embebido: columnas con coeficiente ≠ 0) / `ninguna` |

`mes_sin` y `mes_cos` no se transforman ni escalan (ya están en [-1, 1]). Todo se ajusta solo con la parte de
entrenamiento de cada fold. El RFE usa el mismo modelo que se evalúa, salvo el MLP (no expone coeficientes) que usa Ridge.

Para no recalcular: el pipeline tiene `memory` (un mismo transformador + scaler en el mismo fold se ajusta una sola vez
entre trials) y los scores del filtro (sobre todo la información mutua) también se cachean.

In [ ]:
@memoria.cache
def _scores_filtro(X: pd.DataFrame, y: pd.Series, metodo: str) -> tuple[pd.Series, pd.DataFrame]:
    """|score| de cada columna contra y (ordenado de mayor a menor) y |corr| entre columnas
    (sobre rangos si el método es spearman o mutual_info). Cacheado en disco por (X, y, método)."""
    X = X.loc[:, X.nunique() > 1]                      # fuera constantes en este fold
    y = pd.Series(np.asarray(y, dtype=float), index=X.index)
    R = X.rank() if metodo in ("spearman", "mutual_info") else X
    if metodo == "mutual_info":
        s = pd.Series(mutual_info_regression(X, y, random_state=SEMILLA), index=X.columns)
    else:
        Xr, yr = (R, y.rank()) if metodo == "spearman" else (X, y)
        Xc, yc = Xr - Xr.mean(), yr - yr.mean()
        with np.errstate(invalid="ignore", divide="ignore"):
            s = (Xc.mul(yc, axis=0).sum() / np.sqrt((Xc ** 2).sum() * (yc ** 2).sum())).abs()
    s = s.fillna(0).sort_values(ascending=False)
    with np.errstate(invalid="ignore", divide="ignore"):
        corr = R[s.index].corr().abs().fillna(0)
    return s, corr


class SelectorFiltro(BaseEstimator, TransformerMixin):
    """Top-k por score univariado contra y, descartando columnas redundantes
    (|corr| >= umbral con una ya elegida). umbral=None o >= 1 -> sin control de redundancia."""

    def __init__(self, metodo="spearman", k=80, umbral=0.95):
        self.metodo, self.k, self.umbral = metodo, k, umbral

    def fit(self, X, y):
        s, corr = _scores_filtro(pd.DataFrame(X), pd.Series(np.asarray(y, dtype=float), index=X.index), self.metodo)
        if self.umbral is None or self.umbral >= 1:
            self.columnas_ = s.index[: self.k].tolist()
            return self
        M = corr.to_numpy()
        elegidas = []
        for i in range(len(s)):
            if all(M[i, j] < self.umbral for j in elegidas):
                elegidas.append(i)
                if len(elegidas) == self.k:
                    break
        self.columnas_ = s.index[elegidas].tolist()
        return self

    def transform(self, X):
        return X[self.columnas_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columnas_, dtype=object)


_TRANSFORMADORES = {
    "yeo-johnson": lambda: PowerTransformer(method="yeo-johnson", standardize=False),
    # n_quantiles < filas del fold más chico (~420 con 5 folds)
    "quantile": lambda: QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEMILLA),
}
_SCALERS = {"standard": StandardScaler, "robust": RobustScaler, "minmax": MinMaxScaler}


def construir_pipeline(estimador, columnas, cfg: dict, rfe_estimador=None) -> Pipeline:
    """transformador -> scaler (menos mes_sin/mes_cos) -> filtro -> wrapper/embebido -> modelo."""
    num = [c for c in columnas if c not in COLS_CICLICAS]
    pasos_num = []
    if cfg["transformador"] != "ninguno":
        pasos_num.append(("trans", _TRANSFORMADORES[cfg["transformador"]]()))
    if cfg["scaler"] != "ninguno":
        pasos_num.append(("scaler", _SCALERS[cfg["scaler"]]()))
    prep = ColumnTransformer(
        [("num", Pipeline(pasos_num) if pasos_num else "passthrough", num)],
        remainder="passthrough", verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    sel = cfg["seleccion"]
    filtro = (SelectorFiltro(cfg["filtro_metodo"], cfg["k"], cfg["umbral_redundancia"])
              if sel in ("filtro", "filtro+rfe") else "passthrough")
    if sel == "filtro+rfe":
        wrapper = RFE(rfe_estimador, n_features_to_select=cfg["n_rfe"], step=0.1)
    elif sel == "lasso":
        wrapper = SelectFromModel(Lasso(alpha=cfg["alpha_sel"], max_iter=20_000, random_state=SEMILLA))
    else:
        wrapper = "passthrough"
    pipe = Pipeline([("prep", prep), ("filtro", filtro), ("wrapper", wrapper), ("modelo", estimador)],
                    memory=memoria)
    return pipe.set_output(transform="pandas")


def columnas_elegidas(pipe: Pipeline) -> list[str]:
    """Columnas que llegan al modelo (después de todos los pasos de selección)."""
    return list(pipe[:-1].get_feature_names_out())

## Espacios de búsqueda

In [ ]:
def _mlp(**params) -> TransformedTargetRegressor:
    """Red neuronal (MLP) con el target estandarizado.

    El target está en kW (~10000) y los pesos de la red arrancan cerca de 0: sin estandarizarlo
    la red entrena muy mal. El escalado de y se ajusta solo con los datos de entrenamiento que
    recibe .fit() y viaja dentro del objeto, así que .predict() devuelve kW directamente."""
    return TransformedTargetRegressor(regressor=MLPRegressor(**params), transformer=StandardScaler())


# Optuna no acepta tuplas como opciones categóricas: se elige un nombre y se traduce.
_CAPAS_MLP = {"32": (32,), "64": (64,), "64-32": (64, 32), "128-64": (128, 64)}


def _espacio_ridge(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1e-2, 1e4, log=True)}


def _espacio_lasso(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1e-1, 1e3, log=True)}


def _espacio_xgb(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 200, 800, step=100),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": t.suggest_int("max_depth", 2, 6),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": t.suggest_float("reg_lambda", 1.0, 100.0, log=True),
        "min_child_weight": t.suggest_int("min_child_weight", 1, 5),
    }


def _espacio_mlp(t: optuna.Trial) -> dict:
    return {
        "hidden_layer_sizes": _CAPAS_MLP[t.suggest_categorical("hidden_layer_sizes", list(_CAPAS_MLP))],
        "activation": t.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": t.suggest_float("alpha", 1e-4, 10.0, log=True),   # regularización L2
        "learning_rate_init": t.suggest_float("learning_rate_init", 1e-3, 1e-2, log=True),
        "batch_size": t.suggest_categorical("batch_size", [32, 64, 128]),
    }


ESPACIOS: dict[str, dict] = {
    "ridge": {"clase": Ridge, "fijos": {"random_state": SEMILLA}, "espacio": _espacio_ridge},
    # max_iter acotado: con alpha chico Lasso tarda muchísimo en converger y el MAE decide igual
    "lasso": {"clase": Lasso, "fijos": {"random_state": SEMILLA, "max_iter": 20_000}, "espacio": _espacio_lasso},
    "xgboost": {"clase": XGBRegressor,
                "fijos": {"random_state": SEMILLA, "n_jobs": -1, "tree_method": "hist"},
                "espacio": _espacio_xgb},
    "mlp": {
        "clase": _mlp,
        # Sin early_stopping: scikit-learn lo haría con un 10 % ALEATORIO de train, que en una
        # serie temporal mezcla pasado y futuro. Corta por n_iter_no_change sobre la pérdida de train.
        "fijos": {"random_state": SEMILLA, "solver": "adam", "max_iter": 1000,
                  "n_iter_no_change": 20, "early_stopping": False},
        "espacio": _espacio_mlp,
    },
}


def espacio_prep(t: optuna.Trial) -> dict:
    """Configuración del pipeline (común a los 4 modelos). Los parámetros de cada
    método de selección solo se sugieren si ese método sale elegido."""
    cfg = {
        "transformador": t.suggest_categorical("transformador", ["yeo-johnson", "quantile", "ninguno"]),
        "scaler": t.suggest_categorical("scaler", ["standard", "robust", "minmax", "ninguno"]),
        "seleccion": t.suggest_categorical("seleccion", ["filtro", "filtro+rfe", "lasso", "ninguna"]),
    }
    if cfg["seleccion"] in ("filtro", "filtro+rfe"):
        cfg["filtro_metodo"] = t.suggest_categorical("filtro_metodo", ["pearson", "spearman", "mutual_info"])
        cfg["k"] = t.suggest_int("k", 10, 150, step=10)
        cfg["umbral_redundancia"] = t.suggest_float("umbral_redundancia", 0.80, 1.0)  # 1.0 = sin filtro de redundancia
    if cfg["seleccion"] == "filtro+rfe":
        # RFE siempre después del filtro: correrlo sobre las 390 columnas es carísimo
        cfg["n_rfe"] = max(5, int(cfg["k"] * t.suggest_float("frac_rfe", 0.2, 0.9)))
    if cfg["seleccion"] == "lasso":
        cfg["alpha_sel"] = t.suggest_float("alpha_sel", 1e-1, 1e3, log=True)
    return cfg


def sugerir(t: optuna.Trial, nombre: str) -> tuple[dict, dict]:
    """(hiperparámetros del modelo, configuración del pipeline). Con un FixedTrial reconstruye la mejor."""
    esp = ESPACIOS[nombre]
    return {**esp["fijos"], **esp["espacio"](t)}, espacio_prep(t)


def ajustar(nombre: str, params: dict, cfg: dict, X: pd.DataFrame, y: pd.Series) -> Pipeline:
    clase = ESPACIOS[nombre]["clase"]
    rfe_est = Ridge() if nombre == "mlp" else clase(**params)  # el MLP no tiene coef_ ni feature_importances_
    pipe = construir_pipeline(clase(**params), X.columns, cfg, rfe_est)
    with warnings.catch_warnings():
        # Lasso/MLP avisan "no convergió" y Yeo-Johnson desborda en columnas enormes; el MAE decide igual.
        warnings.simplefilter("ignore", ConvergenceWarning)
        warnings.simplefilter("ignore", RuntimeWarning)
        warnings.simplefilter("ignore", UserWarning)
        return pipe.fit(X, y)

## Registro de experimentos

In [ ]:
COLUMNAS_LOG = [
    "corrida", "timestamp", "modelo",
    "etapa",             # baseline | optuna_cv | optuna_val
    "estado",            # completo | podado (trial de Optuna cortado por el pruner o que falló)
    "hiperparametros",   # del modelo
    "preprocesamiento",  # transformador, scaler y selección
    "mae_cv", "std_cv", "n_folds",             # optuna_cv y baseline (TimeSeriesSplit sobre train)
    "mae_val", "rmse_val", "r2_val", "mae_train",
    "seg_entrenamiento", "n_features", "n_train", "n_val", "semilla", "dataset_sha256",
]


def registrar_experimento(fila: dict, ruta: str | Path = RUTA_LOGS) -> None:
    """Agrega una fila a results/experiment_logs.csv (lo crea si no existe).
    Si el log viene de una versión anterior (otras columnas), lo completa
    con las columnas nuevas sin perder las filas viejas."""
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    nueva = pd.DataFrame([{c: fila.get(c, np.nan) for c in COLUMNAS_LOG}])
    if not ruta.exists():
        nueva.to_csv(ruta, index=False)
        return
    if list(pd.read_csv(ruta, nrows=0).columns) == COLUMNAS_LOG:
        nueva.to_csv(ruta, mode="a", header=False, index=False)
    else:
        previo = pd.read_csv(ruta)
        extra = [c for c in previo.columns if c not in COLUMNAS_LOG]
        pd.concat([previo, nueva], ignore_index=True).reindex(columns=COLUMNAS_LOG + extra).to_csv(ruta, index=False)

## Optimización con Optuna (con pruning)

In [ ]:
def _objetivo(nombre: str, X_tr: pd.DataFrame, y_tr: pd.Series, cv: TimeSeriesSplit,
              anotar: Callable) -> Callable[[optuna.Trial], float]:
    """MAE medio sobre los folds de TimeSeriesSplit (solo train). Después de cada fold reporta
    el MAE medio acumulado; si el pruner decide que el trial no promete, se corta ahí.
    Una config inválida (p. ej. la selección deja 0 columnas) también se poda."""
    def objetivo(trial: optuna.Trial) -> float:
        params, cfg = sugerir(trial, nombre)
        maes: list[float] = []
        n_feat = None
        t0 = time.perf_counter()
        for paso, (i_fit, i_eval) in enumerate(cv.split(X_tr)):
            try:
                pipe = ajustar(nombre, params, cfg, X_tr.iloc[i_fit], y_tr.iloc[i_fit])
                pred = pipe.predict(X_tr.iloc[i_eval])
            except ValueError as e:
                trial.set_user_attr("error", str(e)[:200])
                anotar(params, cfg, maes or [np.nan], time.perf_counter() - t0, "podado", n_feat)
                raise optuna.TrialPruned() from e
            n_feat = len(columnas_elegidas(pipe))
            trial.set_user_attr("n_features", n_feat)
            maes.append(mean_absolute_error(y_tr.iloc[i_eval], pred))
            trial.report(float(np.mean(maes)), step=paso)
            if trial.should_prune():
                anotar(params, cfg, maes, time.perf_counter() - t0, "podado", n_feat)
                raise optuna.TrialPruned()
        anotar(params, cfg, maes, time.perf_counter() - t0, "completo", n_feat)
        return float(np.mean(maes))
    return objetivo


def comparar_modelos(
    modelos: list[str] = MODELOS,
    n_trials: int = N_TRIALS,
    n_splits: int = N_SPLITS,
    timeout: float | None = TIMEOUT,
    ruta_datos: str | Path = RUTA_FINAL,
    ruta_logs: str | Path = RUTA_LOGS,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, optuna.Study], dict[str, Pipeline]]:
    """Baselines + un estudio de Optuna por modelo (TPE + MedianPruner, semilla fija).
    Devuelve (filas registradas en esta corrida, estudios, pipeline final de cada modelo)."""
    desconocidos = set(modelos) - set(ESPACIOS)
    if desconocidos:
        raise ValueError(f"Modelos desconocidos: {sorted(desconocidos)}. Opciones: {list(ESPACIOS)}")

    df = cargar_datos(ruta_datos)  # sin test
    X_tr, y_tr = particion(df, "train")
    X_va, y_va = particion(df, "val")
    sha = huella(ruta_datos)[:12]
    corrida = datetime.now().isoformat(timespec="seconds")
    # test_size fijo: la ventana de entrenamiento más chica es la mitad de train
    cv = TimeSeriesSplit(n_splits=n_splits, test_size=len(X_tr) // (2 * n_splits))
    filas, estudios, finales = [], {}, {}

    def anotar(modelo, etapa, estado, params, cfg, seg, n_feat, **metricas):
        fila = {
            "corrida": corrida, "timestamp": datetime.now().isoformat(timespec="seconds"),
            "modelo": modelo, "etapa": etapa, "estado": estado,
            "hiperparametros": json.dumps(params, sort_keys=True, default=str),
            "preprocesamiento": json.dumps(cfg, sort_keys=True, default=str) if cfg else None,
            "seg_entrenamiento": round(seg, 3), "n_features": n_feat,
            "n_train": len(X_tr), "n_val": len(X_va), "semilla": SEMILLA, "dataset_sha256": sha,
            # redondeo: con n_jobs=-1 el orden de suma cambia decimales lejanos (ruido de punto flotante)
            **{k: (round(v, 6) if isinstance(v, float) else v) for k, v in metricas.items()},
        }
        registrar_experimento(fila, ruta_logs)
        filas.append(fila)

    # --- baselines: mismos folds del CV y mismo val que los modelos
    base = predicciones_baselines(df)
    for nombre in base:
        pred_tr = base.loc[X_tr.index, nombre]
        maes = [_metricas(y_tr.iloc[e], pred_tr.iloc[e])["mae"] for _, e in cv.split(X_tr)]
        m_val = _metricas(y_va, base.loc[X_va.index, nombre])
        anotar(nombre, "baseline", "completo", {}, None, 0.0, None,
               mae_cv=float(np.mean(maes)), std_cv=float(np.std(maes)), n_folds=len(maes),
               mae_val=m_val["mae"], rmse_val=m_val["rmse"], r2_val=m_val["r2"],
               mae_train=_metricas(y_tr, pred_tr)["mae"])
        if verbose:
            print(f"{nombre:26s} MAE cv = {np.mean(maes):8.1f}  MAE val = {m_val['mae']:8.1f}")

    # --- modelos
    for nombre in modelos:
        t_modelo = time.perf_counter()

        def anotar_cv(params, cfg, maes, seg, estado, n_feat, _n=nombre):
            anotar(_n, "optuna_cv", estado, params, cfg, seg, n_feat,
                   mae_cv=float(np.nanmean(maes)) if not np.all(np.isnan(maes)) else np.nan,
                   std_cv=float(np.nanstd(maes)) if not np.all(np.isnan(maes)) else np.nan,
                   n_folds=int(np.sum(~np.isnan(maes))))

        estudio = optuna.create_study(
            direction="minimize",
            study_name=f"{nombre}_{corrida}",
            storage=f"sqlite:///{RUTA_OPTUNA_DB.as_posix()}",
            # multivariate + group: el espacio es condicional (cada selección tiene sus propios parámetros)
            sampler=optuna.samplers.TPESampler(seed=SEMILLA, multivariate=True, group=True),
            # no poda hasta tener 8 trials completos ni antes del 2.º fold
            pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
        )
        estudio.optimize(_objetivo(nombre, X_tr, y_tr, cv, anotar_cv), n_trials=n_trials, timeout=timeout)
        estudios[nombre] = estudio

        # mejor config -> todo train -> validation
        params, cfg = sugerir(optuna.trial.FixedTrial(estudio.best_params), nombre)
        t0 = time.perf_counter()
        pipe = ajustar(nombre, params, cfg, X_tr, y_tr)
        seg = time.perf_counter() - t0
        finales[nombre] = pipe
        m_val = _metricas(y_va, pipe.predict(X_va))
        anotar(nombre, "optuna_val", "completo", params, cfg, seg, len(columnas_elegidas(pipe)),
               mae_val=m_val["mae"], rmse_val=m_val["rmse"], r2_val=m_val["r2"],
               mae_train=_metricas(y_tr, pipe.predict(X_tr))["mae"])

        if verbose:
            estados = [t.state for t in estudio.trials]
            podados = sum(s == optuna.trial.TrialState.PRUNED for s in estados)
            print(f"{nombre:26s} MAE cv = {estudio.best_value:8.1f}  MAE val = {m_val['mae']:8.1f}  "
                  f"({len(estados)} trials, {podados} podados, {time.perf_counter() - t_modelo:.0f} s)")

    return pd.DataFrame(filas), estudios, finales

### Correr el estudio

In [ ]:
res, estudios, finales = comparar_modelos()
print(f"\nLog: {RUTA_LOGS}\nEstudios: {RUTA_OPTUNA_DB}")

## Comparación con los baselines

`mae_cv`: MAE medio en los folds de train (el que optimizó Optuna). `mae_val`: modelo final reentrenado en todo
train y medido en validation. `mejora_*`: cuánto baja el MAE respecto a la persistencia (positivo = le gana).

In [ ]:
def resumen(resultados: pd.DataFrame) -> pd.DataFrame:
    """Modelo final de cada estudio y cada baseline, ordenado por MAE de validación,
    con la mejora (%) respecto a la persistencia."""
    finales_ = resultados[resultados["etapa"].isin(["baseline", "optuna_val"])].set_index("modelo")
    cv = (resultados[resultados["etapa"].isin(["optuna_cv", "baseline"])]
          .groupby("modelo")["mae_cv"].min())
    t = finales_[["mae_val", "rmse_val", "r2_val", "mae_train", "n_features"]].join(cv)
    ref = t.loc["baseline_persistencia"]
    t["mejora_cv_%"] = 100 * (1 - t["mae_cv"] / ref["mae_cv"])
    t["mejora_val_%"] = 100 * (1 - t["mae_val"] / ref["mae_val"])
    cols = ["mae_cv", "mae_val", "mejora_cv_%", "mejora_val_%", "rmse_val", "r2_val", "mae_train", "n_features"]
    return t[cols].sort_values("mae_val")


tabla = resumen(res)
tabla.round(2)

### Mejor configuración de preprocesamiento por modelo

In [ ]:
val = res[res["etapa"] == "optuna_val"].set_index("modelo")
pd.DataFrame({m: json.loads(val.loc[m, "preprocesamiento"]) for m in val.index}).T.fillna("-")

In [ ]:
# MAE de validación por modelo; los baselines van como referencia vertical
TINTA, TINTA_2, MUDO, GRILLA, SERIE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#2a78d6"

modelos_val = tabla.drop(index=[i for i in tabla.index if i.startswith("baseline")]).sort_values("mae_val")
fig, ax = plt.subplots(figsize=(8, 0.6 * len(modelos_val) + 1.6))
ax.barh(modelos_val.index, modelos_val["mae_val"], height=0.5, color=SERIE, zorder=2)
for y_i, v in enumerate(modelos_val["mae_val"]):
    ax.text(v, y_i, f"{v:,.0f}  ", va="center", ha="right", color="#ffffff", fontsize=9, zorder=4)

# etiqueta de cada baseline a la derecha de su línea: una arriba y otra abajo para que no se pisen
n = len(modelos_val)
for (nombre, estilo), y_txt, va in [(("baseline_persistencia", "--"), -0.6, "bottom"),
                                    (("baseline_media_movil_7d", ":"), n - 0.45, "top")]:
    v = tabla.loc[nombre, "mae_val"]
    ax.axvline(v, color=TINTA_2, linestyle=estilo, linewidth=1.5, zorder=1)  # detrás de las barras
    ax.text(v, y_txt, f" {nombre.removeprefix('baseline_').replace('_', ' ')}: {v:,.0f}",
            ha="left", va=va, fontsize=8, color=TINTA_2)
ax.set_ylim(n + 0.1, -0.75)

ax.set_xlabel("MAE en validation (kW) - menor es mejor", color=TINTA_2)
ax.set_xlim(0, max(modelos_val["mae_val"].max(), tabla["mae_val"].max()) * 1.18)
ax.grid(axis="x", color=GRILLA, linewidth=0.8, zorder=0)
ax.tick_params(colors=MUDO, labelcolor=TINTA)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(MUDO)
ax.set_title("Modelos vs baselines", loc="left", color=TINTA, pad=12)
plt.tight_layout()
plt.show()

## Análisis del proceso

Por modelo: historia de la optimización, curvas por fold (las que terminan antes son los trials podados) e
importancia de cada parámetro (¿cuánto pesa el transformador, el scaler o la selección frente a los hiperparámetros?).
Los parámetros condicionales (`k`, `frac_rfe`, `alpha_sel`, ...) se estiman solo con los trials que los usaron: con pocos trials son ruidosos.
El MAE va en escala log: algunas configuraciones extrapolan muy mal y sin log aplastan al resto.

In [ ]:
for nombre, estudio in estudios.items():
    estados = pd.Series([t.state.name for t in estudio.trials]).value_counts().to_dict()
    print(f"=== {nombre}: {estados}")
    # cada función de optuna crea su propia figura y devuelve el Axes
    ax = optuna_plt.plot_optimization_history(estudio)
    ax.set_yscale("log")
    ax.set_title(f"{nombre}: historia (MAE cv)")
    ax = optuna_plt.plot_intermediate_values(estudio)
    ax.set_yscale("log")
    if ax.get_legend():
        ax.get_legend().remove()   # una entrada por trial: con 60 trials no se lee
    ax.set(title=f"{nombre}: MAE acumulado por fold (cortadas = podados)", xlabel="fold")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)  # "pocos trials en algún régimen"
        try:
            ax = optuna_plt.plot_param_importances(estudio)
            ax.set_title(f"{nombre}: importancia de parámetros", loc="left")  # reemplaza el de optuna
        except (ValueError, RuntimeError) as e:  # p. ej. muy pocos trials completos
            print(f"  sin importancias: {e}")
    plt.show()

### Variables que usa cada modelo final

In [ ]:
for nombre, pipe in finales.items():
    cols = columnas_elegidas(pipe)
    total = len(pipe[0].feature_names_in_)
    print(f"\n=== {nombre} ({len(cols)} de {total} variables)")
    print("todas (sin selección)" if len(cols) == total else cols)